In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib

def main():
    print("1. Loading datasets...")
    try:
        users = pd.read_csv('users.csv')
        flights = pd.read_csv('flights.csv')
        hotels = pd.read_csv('hotels.csv')
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Ensure 'users.csv', 'flights.csv', and 'hotels.csv' are in the same directory.")
        return

    print("2. Engineering features...")
    # Aggregate flight data per user
    flights_agg = flights.groupby('userCode').agg(
        num_flights=('travelCode', 'count'),
        avg_flight_price=('price', 'mean')
    ).reset_index()

    # Aggregate hotel data per user
    hotels_agg = hotels.groupby('userCode').agg(
        num_hotels=('travelCode', 'count'),
        avg_hotel_total=('total', 'mean')
    ).reset_index()

    # Merge everything into a single dataset based on the user's code
    df = users.merge(flights_agg, left_on='code', right_on='userCode', how='left')
    df = df.merge(hotels_agg, left_on='code', right_on='userCode', how='left')

    # Define our features (X) and target (y)
    # We drop 'code', 'name', and 'userCode' as they are unique identifiers/names and not useful for generalizing
    X = df[['company', 'age', 'num_flights', 'avg_flight_price', 'num_hotels', 'avg_hotel_total']]
    y = df['gender']

    print("3. Building Machine Learning Pipeline...")
    # Define categorical and numerical columns
    categorical_features = ['company']
    numeric_features = ['age', 'num_flights', 'avg_flight_price', 'num_hotels', 'avg_hotel_total']

    # Create preprocessors for missing values and encoding
    preprocessor = ColumnTransformer(
        transformers=[
            # Fill missing numerical values with 0 (e.g., users with no flights)
            ('num', SimpleImputer(strategy='constant', fill_value=0), numeric_features),
            # Fill missing categorical values with the mode and One-Hot Encode them
            ('cat', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]), categorical_features)
        ])

    # Create the complete pipeline: Preprocessing -> Random Forest Model
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    print("4. Training Model...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    accuracy = model.score(X_test, y_test)
    print(f"Model Training Complete! Validation Accuracy: {accuracy * 100:.2f}%")

    # Save the pipeline to a file so Streamlit can use it
    print("5. Saving model to 'gender_model.joblib'...")
    joblib.dump(model, 'gender_model.joblib')
    print("Done!")

if __name__ == "__main__":
    main()